In [9]:
# let's get what we need together
from __future__ import division
import numpy as np
import numpy.matlib
import matplotlib.pyplot as plt
from jedi.inference.gradual.typing import ProxyTypingClassValue
%matplotlib inline
import pandas
import scipy.io as sio
import brian2
import os
import copy
import pickle

In [10]:
def sigmoid_DA(height,midpoint,slope):
     return np.exp(slope*(height-midpoint))/(1 + np.exp(slope*(height-midpoint)))

In [11]:
#----------------------------------- Parameters ----------------------------------------------#

region_count = 4
population_count = 5
dt = 0.5 * brian2.ms


trial_length = 5        * brian2.second
num_iterations = int(trial_length / dt)

stim_on = 1             * brian2.second
stim_off = 1.4          * brian2.second

#Derivative parameters
a_e = 0.135             * brian2.Hz / brian2.pA
b_e = 54                * brian2.Hz
d_e = 0.308             * brian2.second

c_I_sst = 132           * brian2.Hz / brian2.nA,
c_I_vip = 132           * brian2.Hz / brian2.nA,
c_I_pv = 330            * brian2.Hz / brian2.nA,

r_0_pv = -33            * brian2.Hz
r_0_sst = -33           * brian2.Hz
r_0_vip = -95           * brian2.Hz

r_0_e = 5               * brian2.Hz

#time constants
tau_nmda = 60           * brian2.ms
tau_ampa = 2            * brian2.ms
tau_gaba = 5            * brian2.ms
tau_gaba_dend = 10      * brian2.ms
tau_adapt = 0.1         * brian2.second

gamma_nmda = 1.282      #unitless
gamma_ampa = 5          #unitless
gamma_gaba = 2          #unitless

#defined in the function
# c1 = 120              * brian2.pA
# c2 = 136.24           * brian2.pA
# c3 = 7.0              #unitless
# c4 = 0                * brian2.pA
# c5 = 9.64             * brian2.pA
# c6 = 20               * brian2.pA

#From E                 #unitless
g_e_e = 1
g_dend_e = 0    #no connection
g_pv_e = 1
g_sst_e = 1
g_vip_e = 1

#None from Dendrite here

#From PV                #unitless
g_e_pv_min = 1
g_e_pv_max = 0
g_dend_pv = 0   #no connection
g_pv_pv = 1
g_sst_pv = 0    #no connection
g_vip_pv = 0    #no connection

#From SST
g_e_sst = 0     #no connection
g_dend_sst = 1
g_pv_sst = 1
g_sst_sst = 0   #no connection
g_vip_sst = 0   #no connection

#From VIP
g_e_vip = 0     #no connection
g_dend_vip = 0  #no connection
g_pv_vip = 0    #no connection
g_sst_vip = 1
g_vip_vip = 0   #no connection

g_adapt_e = 1       #temp
g_adapt_dend = 0
g_adapt_pv = 0
g_adapt_sst = 1     #temp
g_adapt_vip = 0     #temp

#AMPA/(AMPA+NMDA) fraction
ampa_frac = 1/6
ampa_frac_pv = 2/6


I_background_e = 310        * brian2.pA
I_background_dend = 300     * brian2.pA
I_background_i = 30         * brian2.pA


#std_noise =

g__m = -0.5                 * brian2.nA

In [12]:
#-------------------------------- Network Initialisation -------------------------------------#

#Long Range Connectivity
struct_connect = np.zeros(region_count)

#Local Connectivity
J = np.array([
    [g_e_e, g_dend_e, g_pv_e, g_sst_e, g_vip_e],
    [0,0,0,0,0],
    [g_e_pv_min, g_dend_pv, g_pv_pv, g_sst_pv, g_vip_pv],
    [g_e_sst, g_dend_sst, g_pv_sst, g_sst_sst, g_vip_sst],
    [g_e_vip, g_dend_vip, g_pv_vip, g_sst_vip, g_vip_vip]
    ]).T * brian2.amp



#Receptor Strength
h1t_1A_strength = np.zeros((1,region_count))
h1t_2A_strength = np.zeros((1,region_count))
h1t_3A_strength = np.zeros((1,region_count))

pops = ['E','Dend','PV','SST','VIP']
pops_column_list  = ['from '+ mystring for mystring in pops]
pops_row_list  = ['to '+ mystring for mystring in pops]

J_display = J*(1/brian2.pA)
df_J = pandas.DataFrame(J_display, columns=pops_column_list, index=pops_row_list)
# print(df_J)

# e_grad = np.ones((region_count, 1))



g_adapt = np.array([g_adapt_e, g_adapt_dend, g_adapt_pv, g_adapt_sst, g_adapt_vip]) * brian2.amp

g_m = np.array([g__m,0,0,0,0]) * brian2.amp

ampa_frac = np.array([ampa_frac,ampa_frac, ampa_frac_pv, ampa_frac, ampa_frac])
nmda_frac = 1 - ampa_frac

J_nmda = J*((J>0).astype(int))
J_ampa = J*((J>0).astype(int))
J_gaba = J*((J<0).astype(int))

J_gaba_dend = np.array([0,0,0,g_e_sst,0]) * brian2.amp

#TEMP
W_superficial = np.zeros(region_count)
W_deep        = np.zeros(region_count)

#TEMP
lr_targets = np.zeros((1, population_count)) * brian2.nA

#No FEF
#No d1r
#No da gradients

In [16]:
#testing shapes

print(nmda_frac.shape)
print(J.shape)
print(J_gaba_dend.shape)

(5,)
(5, 5)
(5,)


In [14]:
#-------------------------------- Local Circuit Object ---------------------------------------#

class LocalCircuit:
    def __init__(self,region,region_number,e_grad):
        self.region = region
        self.region_number = region_number
        #Population Current ---- [ I_total ]
        self.soma_I     = np.zeros((1,num_iterations)) * brian2.pA
        self.dendrite_I = np.zeros((1,num_iterations)) * brian2.pA
        self.pv_I       = np.zeros((1,num_iterations)) * brian2.pA
        self.sst_I      = np.zeros((1,num_iterations)) * brian2.pA
        self.vip_I      = np.zeros((1,num_iterations)) * brian2.pA

        #Firing Rate At Timestep ----- [ R ]
        self.soma_R     = np.zeros((1,num_iterations)) * brian2.Hz
        self.dendrite_R = np.zeros((1,num_iterations)) * brian2.Hz
        self.pv_R       = np.zeros((1,num_iterations)) * brian2.Hz
        self.sst_R      = np.zeros((1,num_iterations)) * brian2.Hz
        self.vip_R      = np.zeros((1,num_iterations)) * brian2.Hz

        #Firing Rate at T0
        self.soma_R[0,0]= r_0_e
        self.pv_R[0,0]  = r_0_e
        self.sst_R[0,0] = r_0_e
        self.vip_R[0,0] = r_0_e

        #temp
        #per pop
        self.I_noise = np.zeros((population_count,1)) * brian2.pA

        #background
        self.I_0 = np.zeros((population_count,1)) * brian2.pA
        self.I_0[0,0] = I_background_e
        self.I_0[1,0] = I_background_dend
        self.I_0[2:,0] = I_background_i

        #temp
        self.e_grad = 1

        #NMDA synapse at soma
        self.S_NMDA     = np.zeros((population_count,num_iterations))

        #AMPA synapse at soma
        self.S_AMPA     = np.zeros((population_count,num_iterations))

        #GABA synapses at soma & dend
        self.S_GABA     = np.zeros((population_count,num_iterations))
        self.S_GABA_DEND     = np.zeros((population_count,num_iterations))

        #S_adapt
        self.S_adapt    = np.zeros((population_count,num_iterations))

        self.I_ext      = np.zeros((population_count,num_iterations))
        #temp
        if region == "first":
            self.I_ext[0,int(stim_on/dt):int(stim_off/dt)] = 1

        #long range
        self.I_lr_nmda = np.zeros((population_count,num_iterations))
        self.I_lr_ampa = np.zeros((population_count,num_iterations))

        #local
        self.I_local_nmda = np.zeros((population_count,num_iterations))
        self.I_local_ampa = np.zeros((population_count,num_iterations))
        self.I_local_gaba = np.zeros((population_count,num_iterations))
        self.I_local_gaba_dend = np.zeros((population_count,num_iterations))

        #temp
        # might go elsewhere
        self.I_exc_dend =
        self.I_inh_dend =
    def lr_update(self,timestep):
        return

    def local_update(self,timestep):
        #I_local_nmda[i_t-1,:,:] = nmda_frac*nmda_da_grad*e_grad*J_nmda.dot(s_nmda[i_t-1,:,:].T).T
        #temp   MISSING SEROTONIN
        self.I_local_nmda[:,timestep-1] = nmda_frac * self.e_grad * J_nmda.dot(self.S_NMDA[:,timestep-1])

        # I_local_ampa[i_t-1,:,:] = ampa_frac*e_grad*J_ampa.dot(s_ampa[i_t-1,:,:].T).T
        #temp   MISSING SEROTONIN
        self.I_local_ampa[:,timestep-1] = ampa_frac * self.e_grad * J_ampa.dot(self.S_AMPA[:,timestep-1])

        # I_local_gaba[i_t-1,:,:] = e_pv_da_mat*(J_gaba.dot(s_gaba[i_t-1,:,:].T).T)
        #temp   MISSING SEROTONIN
        self.I_local_gaba[:,timestep-1] = J_gaba.dot(self.S_GABA[:,timestep-1])

        # I_local_gaba_dend[i_t-1,:,:] = e_sst_da_mat*(J_gaba_dend.dot(s_gaba_dend[i_t-1,:,:].T).T)
        #temp   MISSING SEROTONIN
        self.I_local_gaba_dend[:,timestep-1] = J_gaba_dend * self.S_GABA_DEND[:,timestep-1]

        #SUM Currents
        # I_exc_dend[i_t-1,:,:] = I_local_nmda[i_t-1,:,2:4] + I_lr_nmda[i_t-1,:,2:4] + I_local_ampa[i_t-1,:,2:4] + I_lr_ampa[i_t-1,:,2:4] +I_0[:,2:4] + I_ext[i_t-1,:,2:4] + I_noise[:,2:4]
        I_exc_dend = self.I_local_nmda[1,timestep-1] + self.I_lr_nmda[1,timestep-1] + self.I_local_ampa[1,timestep-1] + self.I_lr_ampa[1,timestep-1] + self.I_0[1,timestep-1] + self.I_ext[1,timestep-1]

        # I_inh_dend[i_t-1,:,:] = I_local_gaba_dend[i_t-1,:,:]
        #temp no reason to create this??
        I_inh_dend[:,timestep-1] = self.I_local_dend[:,timestep-1]

        # I_soma_dend[i_t-1,:,:2]  = dendrite_input_output(I_exc_dend[i_t-1,:,:],I_inh_dend[i_t-1,:,:],parameters)
        self.I_soma_dend = self.dendrite_input_output()


    def current_to_frequency(self,timestep):
        input_current = self.soma_I[0,timestep-1] #for readability
        self.soma_R[0,timestep] =  np.divide((a_e*input_current - b_e),(1 - np.exp(-d_e*(a_e*input_current - b_e))))

        input_current = self.pv_I[0,timestep-1]
        self.pv_R = np.maximum(c_I_pv*input_current + r_0_pv,0)

        input_current = self.sst_I[0,timestep-1]
        self.sst_R = np.maximum(c_I_sst*input_current + r_0_sst,0)

        input_current = self.vip_I[0,timestep-1]
        self.vip_R = np.maximum(c_I_vip*input_current + r_0_vip,0)

    def NMDA_deriv(self,timestep):
        self.S_NMDA[0,timestep] = -(self.S_NMDA[0,timestep-1]/tau_nmda) + gamma_nmda * (1- self.S_NMDA[0,timestep-1]) * self.soma_R[0,timestep]
    def AMPA_deriv(self,timestep):
        self.S_AMPA[0,timestep] = -(self.S_AMPA[0,timestep-1]/tau_ampa) + gamma_ampa * (1- self.S_AMPA[0,timestep-1]) * self.soma_R[0,timestep]

    def GABA_deriv(self,timestep):
        #for soma
        self.S_GABA[0,timestep] = -(self.S_GABA[0,timestep-1]/tau_gaba) + gamma_gaba * self.soma_R[0,timestep]
        #for dendrite
        self.S_GABA[1,timestep] = -(self.S_GABA[1,timestep-1]/tau_gaba) + gamma_gaba * self.soma_R[0,timestep]

    def adaptation_deriv(self,timestep):
        self.S_adapt[:,timestep] = -self.S_adapt[:,timestep-1]/tau_adapt + self.S_adapt[:,timestep]

    def dendrite_input_output(self,timestep):
        c1 = 120              * brian2.pA
        c2 = 136.24           * brian2.pA
        c3 = 7.0              #unitless
        c4 = 0                * brian2.pA
        c5 = 9.64             * brian2.pA
        c6 = 20               * brian2.pA

        beta = c5*np.exp(-self.I_inh_dend[:,timestep-1]/c6)

        return c1*(np.tanh((exc_current +c3*inh_current + c4)/beta)) + c2


In [15]:
circuit1 = LocalCircuit("brain")
# timestep = 1231
# circuit1.region
# circuit1.soma_I[0,timestep-1]

TypeError: LocalCircuit.__init__() missing 2 required positional arguments: 'region_number' and 'e_grad'

In [6]:
J = np.array([
    ["g_e_e",        "g_dend_e",    "g_pv_e",    "g_sst_e",    "g_vip_e"],
    ["0",            "0",           "0",         "0",          "0"],
    ["g_e_pv_min",   "g_dend_pv",   "g_pv_pv",   "g_sst_pv",   "g_vip_pv"],
    ["g_e_sst",      "g_dend_sst",  "g_pv_sst",  "g_sst_sst",  "g_vip_sst"],
    ["g_e_vip",      "g_dend_vip",  "g_pv_vip",  "g_sst_vip",  "g_vip_vip"]
]).T

pops = ['E','Dend','PV','SST','VIP']
pops_column_list  = ['from '+ mystring for mystring in pops]
pops_row_list  = ['to '+ mystring for mystring in pops]

# J_display = J*(1/brian2.pA)
df_J = pandas.DataFrame(J, columns=pops_column_list, index=pops_row_list)

print(df_J)

           from E from Dend     from PV    from SST    from VIP
to E        g_e_e         0  g_e_pv_min     g_e_sst     g_e_vip
to Dend  g_dend_e         0   g_dend_pv  g_dend_sst  g_dend_vip
to PV      g_pv_e         0     g_pv_pv    g_pv_sst    g_pv_vip
to SST    g_sst_e         0    g_sst_pv   g_sst_sst   g_sst_vip
to VIP    g_vip_e         0    g_vip_pv   g_vip_sst   g_vip_vip
